# Notebook 06 — DINO (Deformable DETR) Training

**Dataset:** B (Static/Watchtower) | **Paradigm:** Deformable Transformer  
**Model:** `IDEA-Research/deformable-detr` via HuggingFace

## Config
- Image: 800×800 | Batch: 2 + grad_accum=4 | Epochs: 50 | Optimizer: AdamW lr=0.0002 | FP16: ✅

## Why DINO for Dataset B
Deformable attention uses learned sampling points that naturally conform to **irregular shapes** —
ideal for amorphous smoke plumes at extreme distances. Strongest small-object detector in our lineup.

> **Memory note:** DINO at 800×800 is VRAM-heavy. If OOM: reduce to 640×640 or set `batch=1 + grad_accum=8`.

In [ ]:
# ── Install (Colab) ───────────────────────────────────────────────────────
# !pip install transformers>=4.40.0 pycocotools --quiet

In [ ]:
import torch
from pathlib import Path
from transformers import AutoModelForObjectDetection, AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import json
from PIL import Image

ROOT        = Path('..')
DATA_B      = ROOT / 'data' / 'dataset-b'
RESULTS_DIR = ROOT / 'results' / 'experiment-b' / 'dino'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = 2  # fire, smoke
IMG_SIZE    = (800, 800)  # DINO standard; reduce to (640,640) if OOM
BATCH_SIZE  = 2
GRAD_ACCUM  = 4           # effective batch = 8
print(f"Device: {DEVICE} | Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# ── Load DINO model ───────────────────────────────────────────────────────
MODEL_NAME = "IDEA-Research/deformable-detr"
processor  = AutoImageProcessor.from_pretrained(MODEL_NAME)
model      = AutoModelForObjectDetection.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params/1e6:.1f}M")

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────
class FireDatasetCOCO(Dataset):
    """Reusable COCO-format dataset — same as notebook 05."""
    def __init__(self, img_dir, ann_json, processor, size=(800, 800)):
        self.img_dir   = Path(img_dir)
        self.processor = processor
        self.size      = size
        with open(ann_json) as f:
            coco = json.load(f)
        self.images  = {img['id']: img for img in coco['images']}
        self.img_ids = [img['id'] for img in coco['images']]
        self.anns    = {}
        for ann in coco['annotations']:
            self.anns.setdefault(ann['image_id'], []).append(ann)

    def __len__(self): return len(self.img_ids)

    def __getitem__(self, idx):
        img_id  = self.img_ids[idx]
        img_inf = self.images[img_id]
        image   = Image.open(self.img_dir / img_inf['file_name']).convert('RGB').resize(self.size)
        anns    = self.anns.get(img_id, [])
        boxes   = [[a['bbox'][0], a['bbox'][1], a['bbox'][0]+a['bbox'][2], a['bbox'][1]+a['bbox'][3]] for a in anns]
        labels  = [a['category_id'] for a in anns]
        target  = {'boxes': boxes, 'class_labels': labels, 'image_id': img_id}
        inputs  = self.processor(images=image, annotations=target, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in inputs.items()}

In [ ]:
# ── Training with gradient accumulation ──────────────────────────────────
def collate_fn(batch):
    return {k: [b[k] for b in batch] for k in batch[0]}

train_ds = FireDatasetCOCO(DATA_B/'images'/'train'/'images', DATA_B/'annotations'/'train.json', processor, IMG_SIZE)
val_ds   = FireDatasetCOCO(DATA_B/'images'/'val'/'images',   DATA_B/'annotations'/'val.json',   processor, IMG_SIZE)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
scaler    = GradScaler()
EPOCHS    = 50

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        pixel_values = torch.stack(batch['pixel_values']).to(DEVICE)
        labels = [{k: v.to(DEVICE) for k, v in t.items()} for t in batch['labels']]
        with autocast():
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        total_loss += outputs.loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1} — Loss: {total_loss/len(train_dl):.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
    if (epoch+1) % 10 == 0:
        model.save_pretrained(RESULTS_DIR / f'checkpoint_ep{epoch+1}')

model.save_pretrained(RESULTS_DIR / 'best')
processor.save_pretrained(RESULTS_DIR / 'best')
print("✅ DINO training complete")

In [ ]:
# ── FLOPs measurement ─────────────────────────────────────────────────────
from thop import profile
model.eval()
dummy = torch.zeros(1, 3, 800, 800).to(DEVICE)
try:
    macs, params = profile(model, inputs=(dummy,), verbose=False)
    print(f"Parameters: {params/1e6:.1f}M")
    print(f"GFLOPs:     {macs*2/1e9:.1f}")
except Exception as e:
    print(f"FLOPs via thop failed ({e}) — use model.num_parameters() instead")
    print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")